# SSM Knowledge Distillation Analysis
Analyzing the effect of KD hyperparameters (temperature, alpha) on accuracy and robustness.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

PROJECT_DIR = Path(".")
plt.style.use("seaborn-v0_8-whitegrid")

## 1. Load distillation results

In [ ]:
df_distill = pd.read_csv(PROJECT_DIR / "distillation_results.csv")
print("Distillation results shape:", df_distill.shape)
df_distill

## 2. Distillation accuracy heatmaps

In [ ]:
#average over seeds
df_mean = df_distill.groupby(["temperature", "alpha"]).agg(
    mean_acc=("best_acc", "mean"),
    std_acc=("best_acc", "std"),
    mean_f1=("best_f1", "mean"),
    std_f1=("best_f1", "std"),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes, ["mean_acc", "mean_f1"], ["Mean Accuracy", "Mean Macro F1"]):
    pivot = df_mean.pivot(index="temperature", columns="alpha", values=metric)
    im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("Alpha")
    ax.set_ylabel("Temperature")
    ax.set_title(title)
    ax.grid(False)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, str(round(pivot.values[i, j], 3)),
                    ha="center", va="center", fontsize=10)
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## 3. Accuracy vs alpha (line plot per temperature)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)

for ax, temp in zip(axes, sorted(df_mean["temperature"].unique())):
    subset = df_mean[df_mean["temperature"] == temp]
    ax.errorbar(subset["alpha"], subset["mean_acc"], yerr=subset["std_acc"],
                marker="o", capsize=4, color="tab:blue")
    ax.set_xlabel("Alpha")
    ax.set_title("T=" + str(temp))
    ax.set_xticks(subset["alpha"].values)

axes[0].set_ylabel("Accuracy")
fig.suptitle("Validation Accuracy vs Alpha by Temperature", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Load robustness results

In [ ]:
df_robust = pd.read_csv(PROJECT_DIR / "robustness_results.csv")
print("Robustness results shape:", df_robust.shape)
df_robust

## 5. Robustness: FGSM attack

In [ ]:
df_rob_mean = df_robust.groupby(["temperature", "alpha"]).mean(numeric_only=True).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, eps in zip(axes, [0.01, 0.05, 0.1]):
    col = "fgsm_" + str(eps) + "_acc"
    for temp in sorted(df_rob_mean["temperature"].unique()):
        subset = df_rob_mean[df_rob_mean["temperature"] == temp]
        ax.plot(subset["alpha"], subset[col], marker="o", label="T=" + str(temp))
    ax.set_xlabel("Alpha")
    ax.set_ylabel("Accuracy")
    ax.set_title("FGSM eps=" + str(eps))
    ax.legend()

plt.tight_layout()
plt.show()

## 6. Robustness: Noise injection

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, snr in zip(axes, [20, 10, 5]):
    col = "noise_" + str(snr) + "db_acc"
    for temp in sorted(df_rob_mean["temperature"].unique()):
        subset = df_rob_mean[df_rob_mean["temperature"] == temp]
        ax.plot(subset["alpha"], subset[col], marker="o", label="T=" + str(temp))
    ax.set_xlabel("Alpha")
    ax.set_ylabel("Accuracy")
    ax.set_title("Noise SNR=" + str(snr) + "dB")
    ax.legend()

plt.tight_layout()
plt.show()

## 7. Robustness: Sequence truncation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, pct in zip(axes, [25, 50, 75]):
    col = "trunc_" + str(pct) + "_acc"
    for temp in sorted(df_rob_mean["temperature"].unique()):
        subset = df_rob_mean[df_rob_mean["temperature"] == temp]
        ax.plot(subset["alpha"], subset[col], marker="o", label="T=" + str(temp))
    ax.set_xlabel("Alpha")
    ax.set_ylabel("Accuracy")
    ax.set_title("Truncation " + str(pct) + "%")
    ax.legend()

plt.tight_layout()
plt.show()

## 8. Robustness heatmaps (aggregated)

In [ ]:
#compute average robustness score across all attack types
attack_cols = [
    "fgsm_0.01_acc", "fgsm_0.05_acc", "fgsm_0.1_acc",
    "noise_20db_acc", "noise_10db_acc", "noise_5db_acc",
    "trunc_25_acc", "trunc_50_acc", "trunc_75_acc",
]
df_rob_mean["avg_robust_acc"] = df_rob_mean[attack_cols].mean(axis=1)

#also compute accuracy drop from clean
df_rob_mean["acc_drop"] = df_rob_mean["clean_acc"] - df_rob_mean["avg_robust_acc"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title, cmap in zip(
    axes,
    ["avg_robust_acc", "acc_drop"],
    ["Average Robustness Accuracy", "Accuracy Drop (Clean - Robust)"],
    ["YlOrRd", "YlOrRd_r"]
):
    pivot = df_rob_mean.pivot(index="temperature", columns="alpha", values=metric)
    im = ax.imshow(pivot.values, cmap=cmap, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("Alpha")
    ax.set_ylabel("Temperature")
    ax.set_title(title)
    ax.grid(False)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, str(round(pivot.values[i, j], 3)),
                    ha="center", va="center", fontsize=10)
    fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## 9. Summary comparison: small vs large student

In [ ]:
#load small student results if available
small_path = PROJECT_DIR / "distillation_results_small_student.csv"
if small_path.exists():
    df_small = pd.read_csv(small_path)
    df_small_mean = df_small.groupby(["temperature", "alpha"]).agg(
        mean_acc=("best_acc", "mean")).reset_index()
    df_large_mean = df_distill.groupby(["temperature", "alpha"]).agg(
        mean_acc=("best_acc", "mean")).reset_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    alphas = sorted(df_small_mean["alpha"].unique())
    x = np.arange(len(alphas))
    width = 0.35

    #average over temperatures for simplicity
    small_accs = [df_small_mean[df_small_mean["alpha"] == a]["mean_acc"].mean() for a in alphas]
    large_accs = [df_large_mean[df_large_mean["alpha"] == a]["mean_acc"].mean() for a in alphas]

    ax.bar(x - width/2, small_accs, width, label="Small student (d=32, L=2)")
    ax.bar(x + width/2, large_accs, width, label="Large student (d=64, L=4)")
    ax.set_xticks(x)
    ax.set_xticklabels(alphas)
    ax.set_xlabel("Alpha")
    ax.set_ylabel("Accuracy")
    ax.set_title("Student Size Comparison")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Small student results not found, skipping comparison.")